 # Taller polars y parquet avanzado

**John Gonzalez**

*19 de octubre de 2026*

---

## Objetivo

Imaginen que ustedes trabajan en una empresa que procesan datos bigdata, un ing junior genera código con IA, pero está lleno de problemas. Tu objetivo es corregirlos.

## Entorno de Trabajo

* **Dataset:** Predicción de tarifas de Taxi de Nueva York (5.4 GB en formato CSV, descárgala de kaggle siguiendo las instrucciones vistas en clase, use como referencia el notebook 05).

* **Entorno:** GitHub Codespaces (Límite de recursos: 8 GB RAM).

## A tener en cuenta

* Cuando se le pida "consulte ..." puede apoyarse de la IA para investigar y entender, muy recomendable.
* Cuando se le pida "escriba código" puede apoyarse de la IA, pero debe entender 100% el código generado, se validará en clase el entendimiento, posiblmente se solicite el prompt utilizado para llegar a la solución.
* Cuando se le pida "escriba con sus propias palabras ..." puede parafrasear pero lo escrito debe ser 100% escrito por ud mismo.
* Este taller es la continuación del 07, se recomienda ejecutarlo y entenderlo antes de iniciar.
* Este taller aborda nuevos conceptos, es intencional.

In [7]:
!pip install -q kagglehub


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [4]:
import kagglehub

print("KaggleHub está listo")

KaggleHub está listo


/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
print(bool(os.getenv("KAGGLE_API_TOKEN")))

False


In [7]:
import os

print("¿El token está disponible?:", bool(os.getenv("KAGGLE_API_TOKEN")))

¿El token está disponible?: False


In [11]:
import os
import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass.getpass(
    "Pega aquí tu token nuevo de Kaggle (no se mostrará): "
)

print("Token cargado:", bool(os.environ.get("KAGGLE_API_TOKEN")))

Token cargado: True


In [13]:
import os

token = os.environ.get("KAGGLE_API_TOKEN")

print("¿Hay token cargado?", bool(token))
print("Longitud del token:", len(token) if token else 0)

¿Hay token cargado? True
Longitud del token: 4


In [17]:
import os
import getpass

token = getpass.getpass("Pega tu token completo de Kaggle: ").strip()

if len(token) < 20:
    print("El token parece incompleto. Revisa que hayas copiado el token completo.")
else:
    os.environ["KAGGLE_API_TOKEN"] = token
    print("Token cargado correctamente. Longitud:", len(token))

Token cargado correctamente. Longitud: 37


In [18]:
import kagglehub

path = kagglehub.competition_download(
    "new-york-city-taxi-fare-prediction"
)

print("Datos descargados en:", path)

100%|██████████| 1.56G/1.56G [00:37<00:00, 45.2MB/s]


Extracting files...
Datos descargados en: /home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction


In [20]:
import os

path = "/home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction"

print("Archivos descargados:")
print(os.listdir(path))

Archivos descargados:
['test.csv', 'train.csv', 'sample_submission.csv', 'GCP-Coupons-Instructions.rtf']


In [21]:
import os

path = "/home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction"

archivo_csv = os.path.join(path, "train.csv")
archivo_parquet = os.path.join(path, "train.parquet")

print("¿Existe train.csv?", os.path.exists(archivo_csv))
print("¿Existe train.parquet?", os.path.exists(archivo_parquet))

¿Existe train.csv? True
¿Existe train.parquet? False


In [22]:
import polars as pl

df = pl.scan_csv(archivo_csv)

print("Columnas:")
print(df.collect_schema().names())

print("\nPrimeras 5 filas:")
print(df.head(5).collect())

Columnas:
['key', 'fare_amount', 'pickup_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count']

Primeras 5 filas:
shape: (5, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ key        ┆ fare_amoun ┆ pickup_dat ┆ pickup_lo ┆ pickup_la ┆ dropoff_l ┆ dropoff_l ┆ passenger │
│ ---        ┆ t          ┆ etime      ┆ ngitude   ┆ titude    ┆ ongitude  ┆ atitude   ┆ _count    │
│ str        ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ f64        ┆ str        ┆ f64       ┆ f64       ┆ f64       ┆ f64       ┆ i64       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2009-06-15 ┆ 4.5        ┆ 2009-06-15 ┆ -73.84431 ┆ 40.721319 ┆ -73.84161 ┆ 40.712278 ┆ 1         │
│ 17:26:21.0 ┆            ┆ 17:26:21   ┆ 1         ┆           ┆           ┆           ┆           │
│ 000001

In [19]:
# Configuración del Entorno e Importación de Librerías
import polars as pl
import os
import time

# Rutas de archivos de trabajo
path = "/home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction"
archivo_csv = path + "/train.csv"
archivo_parquet = path + "/train.parquet"

print("Versión de Polars instalada:", pl.__version__)


Versión de Polars instalada: 2.0.0


 ---

 ## Caso 1: Evaluación Ansiosa vs. Perezosa (Gestión de Memoria RAM)



 **Escenario:** Un analista intenta cargar el archivo completo `train.csv` (5.4 GB) a la memoria RAM utilizando evaluación ansiosa (*Eager Execution*) para calcular la tarifa promedio (`fare_amount`). El sistema colapsa debido a un desbordamiento de memoria (*Out of Memory - OOM*).

 > **Tip:** En clase vimos el método `scan_csv`. Puedes ejecutar el cálculo por lotes en disco (*Out-of-Core*) agregando el argumento `streaming=True` al método `.collect()`.

In [1]:
# CÓDIGO INEFICIENTE (CASO 1)
print("--- Caso 1: Intentando lectura ansiosa (Eager) ---")
try:
    inicio = time.time()
    # Ineficiencia: Carga ansiosa que sobrepasa la memoria física disponible
    df = pl.read_csv(archivo_csv)
    media_tarifa = df.select(pl.col("fare_amount").mean())
    print(f"Resultado: {media_tarifa} | Tiempo de ejecución: {time.time() - inicio:.2f} s")
except Exception as e:
    print(f"Error de ejecución detectado (OOM): {e}")


--- Caso 1: Intentando lectura ansiosa (Eager) ---
Error de ejecución detectado (OOM): name 'time' is not defined


 ### [Espacio de Respuesta - Caso 1]

 **Escribe tu código corregido a continuación y documenta:**

 1. El tiempo de ejecución sin streaming con `.collect()`.
 2. El tiempo de ejecución alcanzado con la solución perezosa en modo *streaming* con `.collect(streaming=True)`.
 3. Consulta y explica con tus propias palabras, por qué el procesamiento por lotes (*Out-of-Core*) podría evitar el colapso de la memoria RAM.

In [23]:
# Escribe tu código aquí para el caso 1
print("--- Tiempo de ejecución con .collect() ---")
try:
    inicio = time.time()
    df = pl.scan_csv(archivo_csv)
    media_tarifa = df.select(pl.col("fare_amount").mean()).collect()
    print(f"Resultado: {media_tarifa} | Tiempo de ejecución: {time.time() - inicio:.2f} s")
except Exception as e:
    print(f"Error de ejecución detectado (OOM): {e}")

--- Tiempo de ejecución con .collect() ---
Resultado: shape: (1, 1)
┌─────────────┐
│ fare_amount │
│ ---         │
│ f64         │
╞═════════════╡
│ 11.345046   │
└─────────────┘ | Tiempo de ejecución: 30.62 s


In [24]:
# 2. Evaluación perezosa con streaming
# --------------------------------------------------
print("\n--- Caso 1: Evaluación perezosa con streaming ---")

inicio = time.time()

df = pl.scan_csv(archivo_csv)

media_tarifa_streaming = (
    df.select(pl.col("fare_amount").mean())
    .collect(engine="streaming")
)

tiempo_streaming = time.time() - inicio

print(f"Tarifa promedio: {media_tarifa_streaming}")
print(f"Tiempo con streaming: {tiempo_streaming:.2f} segundos")


--- Caso 1: Evaluación perezosa con streaming ---
Tarifa promedio: shape: (1, 1)
┌─────────────┐
│ fare_amount │
│ ---         │
│ f64         │
╞═════════════╡
│ 11.345046   │
└─────────────┘
Tiempo con streaming: 30.95 segundos


1. Tiempo de ejecución sin streaming: 30,62 segundos.

2. Tiempo de ejecución con streaming: 30,95 segundos.

Ambos métodos calcularon una tarifa promedio de 11.345046. En esta prueba, el método con streaming tardó 0,33 segundos más que el método sin streaming.

3. ¿Por qué el procesamiento por lotes evita el colapso de la memoria RAM?

El procesamiento por lotes permite leer y procesar los datos en partes, en lugar de cargar todo el archivo en la memoria RAM al mismo tiempo. Esto reduce la cantidad de memoria necesaria y facilita trabajar con archivos grandes. Sin embargo, el ahorro de memoria depende de la operación y de cómo se ejecute.

 ---

 ## Caso 2: Inferencia de Tipos y Filtros Temporales (Fechas como String vs. Datetime)



 **Escenario:** Para filtrar las transacciones correspondientes al mes de diciembre de 2015, se aplican expresiones regulares sobre la columna `pickup_datetime` leída como texto (`String`). La consulta requiere escanear millones de cadenas de caracteres, resultando extremadamente lenta.

 > **Tip:** No alcazamos a ver como convertir texto a formato fecha-hora. Pero es clave porque filtrar cadenas de texto en conjuntos de datos masivos inhabilita las optimizaciones vectoriales. Convierte la columna a formato temporal nativo mediante `.str.to_datetime("%Y-%m-%d %H:%M:%S UTC")` o `.cast(pl.Datetime)`. Una vez casteada, utiliza expresiones temporales nativas como `.dt.year() == 2015` y `.dt.month() == 12`.

In [ ]:
# CODIGO INEFICIENTE (CASO 2)
print("--- Caso 2: Búsqueda sobre cadenas de texto ---")
inicio = time.time()

# Ineficiencia: Coincidencia de cadenas de texto sobre 55 millones de registros
resultado_cadenas = (
    pl.scan_csv(archivo_csv)
    .filter(pl.col("pickup_datetime").str.contains("2015-12"))
    .select(pl.len())
    .collect(streaming=True)
)

tiempo_cadenas = time.time() - inicio
print(f"Registros encontrados en Dic 2015: {resultado_cadenas.item(0,0)} | Tiempo: {tiempo_cadenas:.2f} s")


 ### [Espacio de Respuesta - Caso 2]

 **Escribe tu código corregido a continuación y documenta:**

 1. El tiempo de ejecución tras convertir la columna a `Datetime`.

 2. La ganancia porcentual en velocidad frente al escaneo de cadenas de texto.

In [26]:
# Escribe tu código a
import polars as pl
import time

# --------------------------------------------------
# 1. Método original: buscar texto en las fechas
# --------------------------------------------------
print("--- Caso 2: Filtro sobre cadenas de texto ---")

inicio = time.time()

resultado_cadenas = (
    pl.scan_csv(archivo_csv)
    .filter(pl.col("pickup_datetime").str.contains("2015-12"))
    .select(pl.len())
    .collect(engine="streaming")
)

tiempo_cadenas = time.time() - inicio

print(f"Registros encontrados: {resultado_cadenas.item(0, 0)}")
print(f"Tiempo con cadenas: {tiempo_cadenas:.2f} segundos")


# --------------------------------------------------
# 2. Método corregido: convertir a Datetime y filtrar
# --------------------------------------------------
print("\n--- Caso 2: Filtro con Datetime ---")

inicio = time.time()

resultado_fechas = (
    pl.scan_csv(archivo_csv)
    .with_columns(
        pl.col("pickup_datetime").str.to_datetime(
            format="%Y-%m-%d %H:%M:%S UTC",
            strict=False
        )
    )
    .filter(
        (pl.col("pickup_datetime").dt.year() == 2015) &
        (pl.col("pickup_datetime").dt.month() == 12)
    )
    .select(pl.len())
    .collect(engine="streaming")
)

tiempo_fechas = time.time() - inicio

print(f"Registros encontrados: {resultado_fechas.item(0, 0)}")
print(f"Tiempo con Datetime: {tiempo_fechas:.2f} segundos")


# --------------------------------------------------
# 3. Comparar tiempos
# --------------------------------------------------
if resultado_cadenas.item(0, 0) != resultado_fechas.item(0, 0):
    print("\nAdvertencia: los métodos encontraron cantidades diferentes.")
elif tiempo_cadenas > 0:
    ganancia = (
        (tiempo_cadenas - tiempo_fechas) / tiempo_cadenas
    ) * 100

    print(f"\nDiferencia porcentual: {ganancia:.2f}%")

    if ganancia > 0:
        print("El método con Datetime fue más rápido.")
    elif ganancia < 0:
        print("El método con Datetime fue más lento en esta prueba.")
    else:
        print("Ambos métodos tardaron lo mismo.")

--- Caso 2: Filtro sobre cadenas de texto ---
Registros encontrados: 0
Tiempo con cadenas: 31.32 segundos

--- Caso 2: Filtro con Datetime ---
Registros encontrados: 0
Tiempo con Datetime: 36.64 segundos

Diferencia porcentual: -17.00%
El método con Datetime fue más lento en esta prueba.


 ---

 ## Caso 3: Preservación del Plan Lógico de Ejecución (DAG y `.explain()`)



 **Escenario:** En la línea inicial del pipeline, se convierte el `LazyFrame` a `DataFrame` ejecutando `.collect()`. Posteriormente, se aplican los filtros geográficos y la selección de columnas sobre el objeto en memoria.



 > **Recomendación:** Similar a `Spark`, materializar los datos usando `collect` de forma prematura destruye el Grafo Acíclico Dirigido (DAG). Esto impide que el optimizador de Polars aplique *Projection Pushdown* (seleccionar únicamente las columnas requeridas desde el almacenamiento) y *Selection Pushdown* (filtrar filas antes de cargar los datos). Conserva la cadena como `LazyFrame` e inspecciona el plan optimizado invocando `.explain()`.

In [ ]:
# CÓDIGO INEFICIENTE (CASO 3)
print("--- Caso 3: Materialización prematura del DataFrame ---")
inicio = time.time()

# Ineficiencia: Invocación prematura de .collect() que anula las optimizaciones del DAG
# NOTA: Es muy posible que esta linea desborde la memoria y tengas que reiniciar el kernel
df_materializado = pl.scan_parquet(archivo_parquet).collect()

# Las operaciones de filtrado y proyección se ejecutan de forma ansiosa sobre todo el conjunto
resultado_suboptimo = (
    df_materializado
    .filter(pl.col("passenger_count") == 1)
    .select("fare_amount")
)

tiempo_suboptimo = time.time() - inicio
print(f"Tiempo total con materialización prematura: {tiempo_suboptimo:.2f} s")


 ### [Espacio de Respuesta - Caso 3]

 **Escribe tu código corregido a continuación y documenta:**

 1. El código reordenado donde `.collect()` se ubica únicamente al final del procesamiento.

 2. La salida del comando `print(lazy_query.explain())`, señalando dónde el motor aplicó *Selection Pushdown* y *Projection Pushdown*.

In [27]:
# Escribe el código del punto 1 aqui

print("--- Caso 3: Consulta perezosa optimizada ---")
inicio = time.time()

lazy_query = (
    pl.scan_csv(archivo_csv)
    .filter(pl.col("passenger_count") == 1)
    .select("fare_amount")
)

resultado = lazy_query.collect(engine="streaming")

print(f"Tiempo total: {time.time() - inicio:.2f} segundos")
print(resultado.head(5))


--- Caso 3: Consulta perezosa optimizada ---
Tiempo total: 31.72 segundos
shape: (5, 1)
┌─────────────┐
│ fare_amount │
│ ---         │
│ f64         │
╞═════════════╡
│ 4.5         │
│ 16.9        │
│ 7.7         │
│ 5.3         │
│ 12.1        │
└─────────────┘


In [28]:
print(lazy_query.explain())# Escribe el código del punto 2 aqui

simple π 1/1 ["fare_amount"]
  Csv SCAN [/home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction/train.csv]
  PROJECT 2/8 COLUMNS
  SELECTION: col("passenger_count") == 1
  ESTIMATED ROWS: 56971783


 ---

 ## Caso 4: Particionamiento Físico y el "Small File Problem"



 **Escenario:** Con el propósito de organizar un Data Lake en disco, se intenta particionar el conjunto de datos en formato Parquet utilizando la columna de coordenada de longitud (`pickup_longitude`).

En clase vimos que los archivos parquet se dividen en bloques. Esto es muy beneficioso porque cada bloque tiene metadatos como el valor míni y máximo, si por ejemplo tenemos un filtro como `ickup_longitude < -74`, polars leerá solo los bloques que cumplen con esta condición. Además, también existe la posibilidad de dividir los datos en pequeños archivos o carpetas parquet, de tal manera que polars solo se enfocará en leer aquellos archivos que cumplen alguna condición por ejemplo `year=2015`. 


 > **Recomendación:** Polars ofrece la oportunidad de particionar por variable, sin embargo, cuando la partición se hace por variables continuas puede generar miles de directorios y archivos Parquet diminutos (de pocos kilobytes). Esto provoca latencias severas en el sistema de archivos (Disk I/O Latency). Selecciona columnas discretas o categóricas de **baja cardinalidad** (ej. `year` y `month`) para mantener tamaños óptimos (100 MB - 500 MB).

In [ ]:
# CÓDIGO INEFICIENTE (CASO 4)
print("--- Caso 4: Particionamiento por variable continua (Alta Cardinalidad) ---")
# ADVERTENCIA: La siguiente llamada se limita conceptualmente para evitar la creación masiva de archivos.
try:
    # Ineficiencia: Intento de particionamiento sobre una columna flotante continua
    pl.scan_parquet(archivo_parquet).sink_parquet(
        "parquet_inadecuado",
        partition_by=["pickup_longitude"]
    )
except Exception as e:
    print(f"Respuesta del sistema ante particionamiento inadecuado: {e}")


 ### [Espacio de Respuesta - Caso 4]

 **Escribe tu código corregido a continuación y documenta:**

 1. El código que particiona por variables discretas adecuadas (por ejemplo, extraídas de las fechas).

 2. Un comando de terminal o script que verifique la estructura de directorios y el tamaño de los archivos generados. Recuerda que en jupyter para ejecutar un comando de consola se usa el símbolo `!` al inicio de la celda.

In [30]:
# Escribe tu código aquí para el punto 1 aquí

print("--- Caso 4: Particionamiento por año y mes ---")

carpeta_salida = "parquet_particionado"

consulta = (
    pl.scan_csv(archivo_csv)
    .with_columns(
        pl.col("pickup_datetime")
        .str.to_datetime(
            format="%Y-%m-%d %H:%M:%S UTC",
            strict=False
        )
        .alias("pickup_datetime")
    )
    .with_columns(
        pl.col("pickup_datetime").dt.year().alias("year"),
        pl.col("pickup_datetime").dt.month().alias("month")
    )
)

consulta.sink_parquet(
    pl.PartitionBy(
        carpeta_salida,
        key=["year", "month"]
    ),
    mkdir=True,
    engine="streaming"
)

print("Particionamiento finalizado.")



--- Caso 4: Particionamiento por año y mes ---
Particionamiento finalizado.


In [31]:
# Escribe los comandos de term
print("--- Verificación de archivos Parquet ---")

!find parquet_particionado -type f -name "*.parquet" | head -20
!du -sh parquet_particionado
!find parquet_particionado -type f -name "*.parquet" -exec ls -lh {} \; | head -20

--- Verificación de archivos Parquet ---
parquet_particionado/year=2009/month=3/00000000.parquet
parquet_particionado/year=2009/month=12/00000000.parquet
parquet_particionado/year=2009/month=1/00000000.parquet
parquet_particionado/year=2009/month=9/00000000.parquet
parquet_particionado/year=2009/month=6/00000000.parquet
parquet_particionado/year=2009/month=2/00000000.parquet
parquet_particionado/year=2009/month=8/00000000.parquet
parquet_particionado/year=2009/month=10/00000000.parquet
parquet_particionado/year=2009/month=7/00000000.parquet
parquet_particionado/year=2009/month=11/00000000.parquet
parquet_particionado/year=2009/month=4/00000000.parquet
parquet_particionado/year=2009/month=5/00000000.parquet
parquet_particionado/year=2015/month=3/00000000.parquet
parquet_particionado/year=2015/month=1/00000000.parquet
parquet_particionado/year=2015/month=6/00000000.parquet
parquet_particionado/year=2015/month=2/00000000.parquet
parquet_particionado/year=2015/month=4/00000000.parquet
parq

 ---

 ## Caso 5: Resiliencia de Esquemas y Valores Faltantes en Producción



 **Escenario:** Al procesar un nuevo archivo de datos, la inferencia automática de esquemas falla o produce comportamientos inconsistentes debido a la presencia de valores nulos e incoherencias de tipos en la columna `passenger_count`.



 > **Recomendación:** En entornos de producción no se debe confiar ciegamente en la inferencia automática de esquemas sobre archivos planos (CSV). Utiliza la opción `schema_overrides` dentro de `scan_csv()` para definir explícitamente el esquema de datos y aplica expresiones de imputación (`.fill_null()`) o filtrado (`.drop_nulls()`) dentro de la consulta perezosa.

In [ ]:
# CÓDIGO INEFICIENTE (CASO 5)
print("--- Caso 5: Inferencia rígida de esquema sin tratamiento de nulos ---")
try:
    # Ineficiencia: Dependencia de la inferencia de tipos sin manejo de esquema ni nulos
    resultado_esquema = (
        pl.scan_csv(archivo_csv)
        .select(pl.col("passenger_count").mean())
        .collect(streaming=True)
    )
    print("Promedio de pasajeros:", resultado_esquema)
except Exception as e:
    print(f"Error detectado en la inferencia de esquema: {e}")


 ### [Espacio de Respuesta - Caso 5]

 **Escribe tu código corregido a continuación y documenta:**

 1. Consulta como usar `schema_overrides`.
 2. Define explícitamente los tipos usando `schema_overrides`.
 3. Aplica la estrategia para la gestión de valores nulos dentro del marco perezoso.

Escribe con tus propias palabras que es y como usar `schema_overrides`

In [32]:
# Escribe tu código aquí para el print("--- Punto 2: Definición del esquema ---")

df = pl.scan_csv(
    archivo_csv,
    schema_overrides={
        "passenger_count": pl.Int64
    }
)

print(df.collect_schema())

Schema({'key': String, 'fare_amount': Float64, 'pickup_datetime': String, 'pickup_longitude': Float64, 'pickup_latitude': Float64, 'dropoff_longitude': Float64, 'dropoff_latitude': Float64, 'passenger_count': Int64})


In [34]:
# Escribe tu código aquí
# print("--- Punto 3: Tratamiento de valores nulos ---")

resultado = (
    pl.scan_csv(
        archivo_csv,
        schema_overrides={
            "passenger_count": pl.Int64
        }
    )
    .with_columns(
        pl.col("passenger_count").fill_null(0)
    )
    .select(
        pl.col("passenger_count").mean()
    )
    .collect(engine="streaming")
)

print("Promedio de pasajeros:", resultado)

Promedio de pasajeros: shape: (1, 1)
┌─────────────────┐
│ passenger_count │
│ ---             │
│ f64             │
╞═════════════════╡
│ 1.68538         │
└─────────────────┘


schema_overrides es una opción de Polars que permite indicar qué tipo de dato debe tener una columna al leer un archivo CSV. Se usa dentro de scan_csv() para evitar depender únicamente de la detección automática de los tipos de datos.

Por ejemplo, si queremos que la columna passenger_count sea un número entero, escribimos:

In [35]:
pl.scan_csv(
    archivo_csv,
    schema_overrides={"passenger_count": pl.Int64}
)

 ---

 ## Caso 6: Interoperabilidad Eficiente en Machine Learning (Apache Arrow / Zero-Copy)

 **Escenario:** Al preparar la matriz de características para entrenar un modelo predictivo en la librería ara machine learning `Scikit-Learn`, el analista invoca la función `.to_pandas()`, provocando una duplicación innecesaria del consumo de memoria RAM.

 > **Recomendación:** Polars gestiona sus datos internamente mediante el formato de memoria **Apache Arrow**. La conversión a Pandas genera una copia completa en memoria bajo bloques de NumPy. Para pasar datos a algoritmos de Machine Learning de forma eficiente, utiliza la exportación nativa a tablas Arrow (`.to_arrow()`) o arreglos NumPy directos, aprovechando la transferencia sin copia (*Zero-Copy*).

In [ ]:
# CÓDIGO INEFICIENTE (CASO 6)
print("--- Caso 6: Duplicación de memoria al convertir a Pandas ---")
inicio = time.time()

# Ineficiencia: Carga a Polars y posterior conversión ansiosa a Pandas
df_polars = pl.scan_parquet(archivo_parquet).select(["fare_amount", "passenger_count"]).collect()
df_pandas = df_polars.to_pandas()  # Genera duplicación de datos en RAM

tiempo_conversion = time.time() - inicio
print(f"Matriz procesada y convertida a Pandas en {tiempo_conversion:.2f} s")


 ### [Espacio de Respuesta - Caso 6]

 **Escribe tu código corregido a continuación y documenta:**

 1. Escribe el código para la exportación a Apache Arrow o arreglos NumPy directa.
 2. Explica conceptualmente con tus propias palabras cómo la especificación de memoria de Apache Arrow evita la duplicación de bytes en la RAM.

In [36]:
# Escribe tu código aquí
print("--- Caso 6: Exportación a Apache Arrow ---")

inicio = time.time()

df_polars = (
    pl.scan_csv(archivo_csv)
    .select(["fare_amount", "passenger_count"])
    .collect(engine="streaming")
)

df_arrow = df_polars.to_arrow()

tiempo_conversion = time.time() - inicio

print(f"Conversión finalizada en {tiempo_conversion:.2f} segundos")
print(df_arrow.schema)
print(df_arrow.slice(0, 5))


--- Caso 6: Exportación a Apache Arrow ---
Conversión finalizada en 32.15 segundos
fare_amount: double
passenger_count: int64
pyarrow.Table
fare_amount: double
passenger_count: int64
----
fare_amount: [[4.5,16.9,5.7,7.7,5.3]]
passenger_count: [[1,1,2,1,1]]


Explica el punto 2 aquí con tus propias palabras

Apache Arrow es una forma de organizar los datos en la memoria del computador para que diferentes herramientas puedan compartirlos de manera eficiente. Cuando Polars exporta datos a Arrow, puede reutilizar los mismos bloques de memoria en lugar de copiar todos los datos a otra estructura. Esto reduce el consumo de RAM y hace más eficiente el intercambio de información. 

---

Felicitaciones, haz terminado el taller, ahora entiendes los conceptos claves de cualquier ingeniero de datos que trabaja en una empresa mediana-grande.